In [0]:
from pyspark.sql import functions as F, Window

CAT = "workspace"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CAT}.silver")

DataFrame[]

In [0]:
df = spark.table(f"{CAT}.bronze.tb_movies_info")

# o bronze usa append, que aí o mesmo filme pode aparecer várias vezes.
df = df.withColumn("id", F.trim(F.col("id").cast("string"))).filter(F.col("id").isNotNull() & (F.col("id") != ""))

w = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())
df = df.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")

In [0]:
# o status vem com ruído. Normaliza primeiro e só depois traduz
# o que não tiver no mapa vira "Não Informado"

# normaliza
df = df.withColumn(
    "status_norm",
    F.trim(F.regexp_replace(
        F.regexp_replace(
            F.regexp_replace(F.lower(F.col("status")), r"[-_]+", " "),
            r"[^a-z ]", ""),
        r"\s+", " "))
)

# traduz
mapa_status = {
    "released": "Lançado",
    "post production": "Pós-Produção",
    "in production": "Em Produção",
    "planned": "Planejado",
    "rumored": "Rumores",
    "canceled": "Cancelado",
    "cancelled": "Cancelado",
}
status_pt = F.lit("Não Informado")
for k, v in mapa_status.items():
    status_pt = F.when(F.col("status_norm") == k, F.lit(v)).otherwise(status_pt)

df = df.withColumn("status_filme", status_pt)

In [0]:
# a data vem em 3 padrões
# try_to_date devolve NULL em vez de erro, então só vira NULL o que não tem conversão possível
formatos = ["yyyy-MM-dd",
            "dd/MM/yyyy",
            "MM-dd-yyyy"]

data_lancamento = F.coalesce(*[F.expr(f"try_to_date(trim(release_date), '{f}')") for f in formatos])

silver_info = (df
    .withColumn("data_lancamento", data_lancamento)
    .withColumn("ano_lancamento", F.year("data_lancamento"))
    .withColumn("duracao_minutos", F.expr("try_cast(try_cast(trim(runtime) as double) as int)"))
    .select(
        F.col("id").alias("id_filme"),
        F.col("title").alias("titulo"),
        F.col("original_title").alias("titulo_original"),
        "data_lancamento",
        "duracao_minutos",
        F.col("original_language").alias("idioma_original"),
        "status_filme",
        F.col("overview").alias("sinopse"),
        F.col("tagline").alias("frase_divulgacao"),
        "ano_lancamento",
    ))

(silver_info.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CAT}.silver.tb_info_filmes"))

In [ ]:
# conferências da tb_info_filmes
s = spark.table(f"{CAT}.silver.tb_info_filmes")
print("linhas:", s.count(), "| ids únicos:", s.select("id_filme").distinct().count())
s.groupBy("status_filme").count().show()

print("datas nulas:", s.filter("data_lancamento is null").count())

# valores originais que não converteram
chk = df.withColumn("data_lancamento", data_lancamento)
(chk.filter("data_lancamento is null and release_date is not null")
    .select("release_date").distinct().show(50, False))

# padrões de data que existem na base
(df.withColumn("padrao", F.regexp_replace(F.regexp_replace(F.trim("release_date"), r"[0-9]", "9"), r"[A-Za-z]", "a"))
   .groupBy("padrao").count().orderBy(F.desc("count")).show(20, False))

In [ ]:
# olhar como a cotação ficou na bronze antes de tratar
b = spark.table(f"{CAT}.bronze.tb_cotacao_dolar")
b.printSchema()
b.orderBy("dataHoraCotacao").show(15, False)
print("linhas:", b.count())

In [ ]:
# a API do BC só tem cotação em dia útil. a série precisa ser contínua
# fim de semana e feriado recebem a cotação do último dia útil
cot = spark.table(f"{CAT}.bronze.tb_cotacao_dolar")

cot = (cot
    .withColumn("data_cotacao", F.expr("try_to_date(substring(trim(dataHoraCotacao), 1, 10), 'yyyy-MM-dd')"))
    .withColumn("cotacao_dolar", F.expr("try_cast(cotacaoCompra as decimal(18,6))"))
    .filter("data_cotacao is not null and cotacao_dolar is not null and cotacao_dolar > 0"))

# a API devolve mais de um boletim por dia: fica só o último do dia
w_dia = Window.partitionBy("data_cotacao").orderBy(F.col("dataHoraCotacao").desc(), F.col("ingestion_datetime").desc())
cot = cot.withColumn("rn", F.row_number().over(w_dia)).filter("rn = 1").select("data_cotacao", "cotacao_dolar")

# calendário contínuo entre a primeira e a última cotação
calendario = (cot.agg(F.min("data_cotacao").alias("ini"), F.max("data_cotacao").alias("fim"))
    .select(F.explode(F.sequence("ini", "fim", F.expr("interval 1 day"))).alias("data_cotacao")))

# forward fill: repete o último valor não nulo
w_ff = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, Window.currentRow)
silver_cot = (calendario.join(cot, "data_cotacao", "left")
    .withColumn("cotacao_dolar", F.last("cotacao_dolar", ignorenulls=True).over(w_ff)))

(silver_cot.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CAT}.silver.tb_cotacao_dolar"))

spark.table(f"{CAT}.silver.tb_cotacao_dolar").orderBy("data_cotacao").show(15)

In [ ]:
# budget/revenue vêm sujos
# limpa primeiro e só depois converte pra decimal. texto sem número vira NULL
fin = spark.table(f"{CAT}.bronze.tb_movies_financials")
fin = fin.withColumn("id", F.trim(F.col("id").cast("string"))).filter(F.col("id").isNotNull() & (F.col("id") != ""))

for origem, destino in [("budget", "orcamento_usd"), ("revenue", "receita_usd")]:
    fin = (fin
        .withColumn("_s", F.upper(F.trim(F.col(origem).cast("string"))))
        .withColumn("_num", F.regexp_replace("_s", r"[^0-9.\-]", ""))
        .withColumn(destino,
            F.expr("try_cast(_num as decimal(20,2))")
            * F.when(F.col("_s").rlike(r"^[0-9.]+M$"), 1000000).otherwise(1))   # 34.0M = 34 milhões
        # valor zerado ou negativo não faz sentido pra orçamento/receita: vira NULL
        .withColumn(destino, F.when(F.col(destino) > 0, F.col(destino).cast("decimal(18,2)")))
        .drop("_s", "_num"))

# 1 linha por filme. ingestion_datetime é igual nas repetidas, então desempata pela linha mais completa
completude = F.col("orcamento_usd").isNotNull().cast("int") + F.col("receita_usd").isNotNull().cast("int")
w = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc(), completude.desc(),
                                     F.col("orcamento_usd").desc_nulls_last(), F.col("receita_usd").desc_nulls_last())
fin = fin.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")

# regra: conversão pra BRL com a cotação mais recente da silver
cotacao = spark.table(f"{CAT}.silver.tb_cotacao_dolar").orderBy(F.col("data_cotacao").desc()).first()["cotacao_dolar"]
print("cotação usada:", cotacao)

silver_fin = (fin
    .withColumn("orcamento_brl", F.round(F.col("orcamento_usd") * F.lit(cotacao), 2).cast("decimal(18,2)"))
    .withColumn("receita_brl", F.round(F.col("receita_usd") * F.lit(cotacao), 2).cast("decimal(18,2)"))
    # lucro: valor ausente conta como 0 pra não anular a conta; só fica NULL se orçamento E receita faltam
    .withColumn("lucro_usd",
        F.when(F.col("orcamento_usd").isNull() & F.col("receita_usd").isNull(), F.lit(None).cast("decimal(18,2)"))
         .otherwise((F.coalesce("receita_usd", F.lit(0)) - F.coalesce("orcamento_usd", F.lit(0))).cast("decimal(18,2)")))
    .withColumn("lucro_brl", F.round(F.col("lucro_usd") * F.lit(cotacao), 2).cast("decimal(18,2)"))
    # margem %: só calcula com receita > 0, o que evita divisão por zero
    .withColumn("margem_lucro_percentual",
        F.when(F.col("receita_usd") > 0, F.round(F.col("lucro_usd") / F.col("receita_usd") * 100, 2).cast("double")))
    .select(F.col("id").alias("id_filme"), "orcamento_usd", "receita_usd", "orcamento_brl", "receita_brl",
            "lucro_usd", "lucro_brl", "margem_lucro_percentual"))

(silver_fin.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CAT}.silver.tb_financeiro_filmes"))

In [ ]:
# conferências da tb_financeiro_filmes
s = spark.table(f"{CAT}.silver.tb_financeiro_filmes")
print("linhas:", s.count(), "| ids únicos:", s.select("id_filme").distinct().count())
s.printSchema()

# quantos nulos em cada coluna
s.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in s.columns]).show()

# os casos que a gente viu na bronze (duplicado com $, N/A e 0)
s.filter(F.col("id_filme").isin("382589", "524369", "744276")).show(truncate=False)

s.orderBy(F.col("receita_usd").desc_nulls_last()).show(5, False)